# SignBridge — Notebook 4: BiLSTM with Attention Temporal Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/joshiaradhya/SignBridge/blob/main/notebooks/04_temporal_training.ipynb)

This notebook:
1. Loads the preprocessed landmark sequence datasets (`.npz`).
2. Configures the **BiLSTM with Temporal Attention** architecture.
3. Executes resumable GPU training with learning rate scheduling and early stopping.
4. Performs held-out signer-independent test evaluation (Top-1, Top-5, Macro F1, Confusion Matrix).
5. Saves persistent checkpoints for ISL and ASL separately.

In [ ]:
# Step 1: Environment Setup
!git clone https://github.com/joshiaradhya/SignBridge.git /content/SignBridge || (cd /content/SignBridge && git pull)
%cd /content/SignBridge
!pip install -q -r ml/colab-requirements.txt

In [ ]:
# Step 2: Train Indian Sign Language (ISL) model
from ml.signbridge.training import SignModelTrainer, TrainingConfig, evaluate_test_set

isl_config = TrainingConfig(
    input_dim=222,
    hidden_dim=128,
    lstm_hidden=128,
    num_layers=2,
    dropout=0.3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-3,
    patience=15,
    device='cuda'
)

trainer_isl = SignModelTrainer(isl_config, output_dir='runs/isl', checkpoint_dir='checkpoints/isl')
npz_isl = 'data/isl_features_v2.npz'

try:
    trainer_isl.fit(npz_isl)
    # Evaluate on held-out test signers
    test_metrics = evaluate_test_set('checkpoints/isl/best_model.pt', npz_isl, 'runs/isl/eval')
except Exception as e:
    print('Run extraction in Notebook 3 before training:', e)

In [ ]:
# Step 3: Train American Sign Language (ASL) model
asl_config = TrainingConfig(
    input_dim=222,
    hidden_dim=128,
    lstm_hidden=128,
    num_layers=2,
    dropout=0.3,
    epochs=50,
    batch_size=32,
    learning_rate=1e-3,
    patience=15,
    device='cuda'
)

trainer_asl = SignModelTrainer(asl_config, output_dir='runs/asl', checkpoint_dir='checkpoints/asl')
npz_asl = 'data/asl_features_v2.npz'

try:
    trainer_asl.fit(npz_asl)
    # Evaluate on held-out test signers
    test_metrics = evaluate_test_set('checkpoints/asl/best_model.pt', npz_asl, 'runs/asl/eval')
except Exception as e:
    print('Run extraction in Notebook 3 before training:', e)